# Steady-State Simulation: Voltage Profile Analysis## ObjectiveAssesses voltage level variations caused by new generation integrationProject: 39 Bus New England System - 2**Study Case: Study Cases 1. Power Flow****Objective:**- Assess voltage level variations caused by new generation integration- Extract bus voltage magnitudes for both scenarios- Compare voltage levels using a radar (spider) plot- Outputs: Radar plot of bus voltages, Voltage deviation comparison between cases---

## Step 1: Access PowerFactoryFirst, we need to set up the Python environment to access DIgSILENT PowerFactory.

In [ ]:
# ============================================================================# STEP 1: Access PowerFactory# ============================================================================import osos.environ["PATH"] = r"C:\Program Files\DIgSILENT\PowerFactory 2021 SP2" + os.environ["PATH"]import syssys.path.append(r"C:\Program Files\DIgSILENT\PowerFactory 2021 SP2\Python\3.9")# Import powerfactoryimport powerfactory as pfapp = pf.GetApplication()  # Get the application# ============================================================================# STEP 2: Access and activate project# ============================================================================user = app.GetCurrentUser()project = app.ActivateProject("39 Bus New England System - 2")  # Activate the desired projectprj = app.GetActiveProject()print(f"Project activated: {prj.loc_name}")# ============================================================================# STEP 2.5: Activate study case (if needed)# ============================================================================# Try to activate the study case "Study Cases 1. Power Flow"try:    study_cases = prj.GetContents('*.IntCase')    for sc in study_cases:        if '1. Power Flow' in sc.loc_name or 'Power Flow' in sc.loc_name:            sc.Activate()            print(f"Study case activated: {sc.loc_name}")            breakexcept:    print("Note: Using default/active study case")# ============================================================================# STEP 3: Get all relevant objects (buses)# ============================================================================# Create bus dictionarybuses = app.GetCalcRelevantObjects('*.ElmTerm')bus_dict = {}for bus in buses:    bus_dict[bus.loc_name] = busprint(f"Found {len(bus_dict)} buses")# ============================================================================# STEP 4: Run Load Flow Analysis - Base Case# ============================================================================print("\n=== Running Load Flow Analysis - Base Case ===")app.ResetCalculation()# Get load flow calculation commandloadflow = app.GetFromStudyCase('ComLdf')loadflow.iopt_net = 0  # Balanced 3-phase calculationloadflow.Execute()print("Load flow calculation completed for Base Case")# ============================================================================# STEP 5: Extract Base Case Voltage Profile# ============================================================================base_case_voltages = {}for bus_name, bus in bus_dict.items():    v_mag = bus.GetAttribute('m:u')  # Voltage magnitude in p.u.    base_case_voltages[bus_name] = v_magprint(f"Extracted voltage profile for {len(base_case_voltages)} buses (Base Case)")# ============================================================================# STEP 6: New Generation Case (if applicable)# ============================================================================print("\n=== Running Load Flow Analysis - New Generation Case ===")# NOTE: This section should be modified based on how new generation is added# For now, we'll re-run load flow (assuming new generation is already added)app.ResetCalculation()loadflow.Execute()print("Load flow calculation completed for New Generation Case")# Extract New Generation Case Voltage Profilenew_gen_voltages = {}for bus_name, bus in bus_dict.items():    v_mag = bus.GetAttribute('m:u')  # Voltage magnitude in p.u.    new_gen_voltages[bus_name] = v_magprint(f"Extracted voltage profile for {len(new_gen_voltages)} buses (New Generation Case)")# ============================================================================# STEP 7: Calculate Voltage Deviations# ============================================================================voltage_deviations = {}for bus_name in bus_dict.keys():    if bus_name in base_case_voltages and bus_name in new_gen_voltages:        deviation = new_gen_voltages[bus_name] - base_case_voltages[bus_name]        deviation_pct = (deviation / base_case_voltages[bus_name]) * 100 if base_case_voltages[bus_name] != 0 else 0        voltage_deviations[bus_name] = {            'base_case': base_case_voltages[bus_name],            'new_gen': new_gen_voltages[bus_name],            'deviation': deviation,            'deviation_pct': deviation_pct        }# ============================================================================# STEP 8: Export Results to CSV# ============================================================================import csvimport osscript_dir = os.path.dirname(os.path.abspath(__file__))# Export voltage comparisonvoltage_csv_path = os.path.join(script_dir, 'voltage_profile_comparison.csv')with open(voltage_csv_path, 'w', newline='') as csvfile:    writer = csv.writer(csvfile)    writer.writerow(['Bus Name', 'Base Case Voltage (p.u.)', 'New Gen Voltage (p.u.)',                      'Deviation (p.u.)', 'Deviation (%)'])    for bus_name, data in voltage_deviations.items():        writer.writerow([bus_name, data['base_case'], data['new_gen'],                         data['deviation'], data['deviation_pct']])print(f"Voltage profile comparison exported to: voltage_profile_comparison.csv")# ============================================================================# STEP 9: Load CSV Data and Create Visualizations# ============================================================================print("\n=== Creating Visualizations ===")try:    import pandas as pd    import matplotlib.pyplot as plt    import seaborn as sns    import numpy as np    from bokeh.plotting import figure, output_file, save    from bokeh.models import ColumnDataSource, HoverTool    from bokeh.layouts import gridplot        # Set style    sns.set_style("whitegrid")    plt.rcParams['figure.figsize'] = (14, 10)        # Load CSV data    voltage_df = pd.read_csv(voltage_csv_path)        # 1. Radar Plot (Spider Plot)    bus_names = voltage_df['Bus Name'].head(20).tolist()  # Limit to 20 buses    base_values = voltage_df['Base Case Voltage (p.u.)'].head(20).tolist()    new_gen_values = voltage_df['New Gen Voltage (p.u.)'].head(20).tolist()        if len(bus_names) > 0:        angles = np.linspace(0, 2 * np.pi, len(bus_names), endpoint=False).tolist()        angles += angles[:1]                base_values += base_values[:1]        new_gen_values += new_gen_values[:1]                fig, axes = plt.subplots(2, 2, figsize=(16, 14))                # Radar plot        ax_radar = plt.subplot(2, 2, 1, projection='polar')        ax_radar.plot(angles, base_values, 'o-', linewidth=2, label='Base Case', color='blue')        ax_radar.plot(angles, new_gen_values, 'o-', linewidth=2, label='New Generation Case', color='red')        ax_radar.fill(angles, base_values, alpha=0.25, color='blue')        ax_radar.fill(angles, new_gen_values, alpha=0.25, color='red')        ax_radar.set_xticks(angles[:-1])        ax_radar.set_xticklabels(bus_names, fontsize=7)        ax_radar.set_ylim(0.9, 1.1)        ax_radar.set_yticks([0.9, 0.95, 1.0, 1.05, 1.1])        ax_radar.set_yticklabels(['0.9', '0.95', '1.0', '1.05', '1.1'], fontsize=8)        ax_radar.grid(True)        ax_radar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))        ax_radar.set_title('Voltage Profile Comparison (Radar Plot)', size=12, fontweight='bold', pad=20)                # Voltage comparison bar plot        voltage_df_sorted = voltage_df.sort_values('Base Case Voltage (p.u.)', ascending=False).head(20)        x_pos = np.arange(len(voltage_df_sorted))        width = 0.35        axes[0, 1].bar(x_pos - width/2, voltage_df_sorted['Base Case Voltage (p.u.)'],                       width, label='Base Case', color='blue', alpha=0.7)        axes[0, 1].bar(x_pos + width/2, voltage_df_sorted['New Gen Voltage (p.u.)'],                       width, label='New Generation Case', color='red', alpha=0.7)        axes[0, 1].axhline(y=1.0, color='black', linestyle='--', linewidth=1, alpha=0.5, label='Nominal')        axes[0, 1].set_xlabel('Bus Name', fontsize=10)        axes[0, 1].set_ylabel('Voltage (p.u.)', fontsize=10)        axes[0, 1].set_title('Voltage Comparison: Top 20 Buses', fontsize=12, fontweight='bold')        axes[0, 1].set_xticks(x_pos)        axes[0, 1].set_xticklabels(voltage_df_sorted['Bus Name'], rotation=45, ha='right', fontsize=7)        axes[0, 1].legend()        axes[0, 1].grid(True, alpha=0.3, axis='y')                # Voltage deviation plot        axes[1, 0].scatter(voltage_df['Base Case Voltage (p.u.)'],                           voltage_df['Deviation (p.u.)'],                           c=voltage_df['Deviation (%)'], cmap='RdYlGn',                           s=60, alpha=0.6, edgecolors='black', linewidths=0.5)        axes[1, 0].axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)        axes[1, 0].set_xlabel('Base Case Voltage (p.u.)', fontsize=10)        axes[1, 0].set_ylabel('Voltage Deviation (p.u.)', fontsize=10)        axes[1, 0].set_title('Voltage Deviation Analysis', fontsize=12, fontweight='bold')        cbar = plt.colorbar(axes[1, 0].collections[0], ax=axes[1, 0])        cbar.set_label('Deviation (%)', fontsize=9)        axes[1, 0].grid(True, alpha=0.3)                # Deviation percentage distribution        axes[1, 1].hist(voltage_df['Deviation (%)'], bins=30, color='steelblue',                        alpha=0.7, edgecolor='black', linewidth=0.5)        axes[1, 1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='No Change')        axes[1, 1].set_xlabel('Voltage Deviation (%)', fontsize=10)        axes[1, 1].set_ylabel('Number of Buses', fontsize=10)        axes[1, 1].set_title('Voltage Deviation Distribution', fontsize=12, fontweight='bold')        axes[1, 1].legend()        axes[1, 1].grid(True, alpha=0.3, axis='y')                plt.tight_layout()        plot_path = os.path.join(script_dir, 'voltage_profile_analysis_plots.png')        plt.savefig(plot_path, dpi=300, bbox_inches='tight')        plt.close()        print(f"Static plots saved to: voltage_profile_analysis_plots.png")                # 2. Interactive Bokeh Plot        try:            output_file(os.path.join(script_dir, 'voltage_profile_interactive.html'))                        # Voltage comparison interactive plot            p1 = figure(width=900, height=500, title="Voltage Profile Comparison (Interactive)",                       x_axis_label="Bus Name", y_axis_label="Voltage (p.u.)",                       tools="pan,wheel_zoom,box_zoom,reset,hover,save",                       x_range=voltage_df_sorted['Bus Name'].head(30).tolist())                        source = ColumnDataSource(data=dict(                bus_names=voltage_df_sorted['Bus Name'].head(30),                base_voltages=voltage_df_sorted['Base Case Voltage (p.u.)'].head(30),                new_gen_voltages=voltage_df_sorted['New Gen Voltage (p.u.)'].head(30),                deviations=voltage_df_sorted['Deviation (p.u.)'].head(30),                deviation_pct=voltage_df_sorted['Deviation (%)'].head(30)            ))                        p1.vbar(x='bus_names', top='base_voltages', width=0.4, source=source,                    color='blue', alpha=0.7, legend_label='Base Case')            p1.vbar(x='bus_names', top='new_gen_voltages', width=0.4, source=source,                    color='red', alpha=0.7, x_offset=0.4, legend_label='New Generation Case')            p1.line([voltage_df_sorted['Bus Name'].head(30).iloc[0],                     voltage_df_sorted['Bus Name'].head(30).iloc[-1]],                    [1.0, 1.0], color='black', line_dash='dashed', line_width=2,                    legend_label='Nominal (1.0 p.u.)')                        hover = p1.select_one(HoverTool)            hover.tooltips = [("Bus", "@bus_names"),                              ("Base Case", "@base_voltages{0.000} p.u."),                             ("New Gen", "@new_gen_voltages{0.000} p.u."),                             ("Deviation", "@deviations{0.000} p.u."),                             ("Deviation %", "@deviation_pct{0.00}%")]                        p1.xaxis.major_label_orientation = np.pi/4            p1.legend.location = "top_right"                        # Deviation scatter plot            p2 = figure(width=900, height=400, title="Voltage Deviation Analysis (Interactive)",                       x_axis_label="Base Case Voltage (p.u.)", y_axis_label="Deviation (p.u.)",                       tools="pan,wheel_zoom,box_zoom,reset,hover,save")                        dev_source = ColumnDataSource(data=dict(                x=voltage_df['Base Case Voltage (p.u.)'],                y=voltage_df['Deviation (p.u.)'],                bus_names=voltage_df['Bus Name'],                deviation_pct=voltage_df['Deviation (%)']            ))                        p2.circle('x', 'y', size=8, source=dev_source, color='steelblue', alpha=0.6)            p2.line([0.9, 1.1], [0, 0], color='red', line_dash='dashed', line_width=2)                        hover2 = p2.select_one(HoverTool)            hover2.tooltips = [("Bus", "@bus_names"),                               ("Base Voltage", "@x{0.000} p.u."),                              ("Deviation", "@y{0.000} p.u."),                              ("Deviation %", "@deviation_pct{0.00}%")]                        grid = gridplot([[p1], [p2]], toolbar_location='right')            save(grid)            print(f"Interactive plots saved to: voltage_profile_interactive.html")        except Exception as e:            print(f"Note: Bokeh interactive plot creation failed: {e}")            print("Install bokeh for interactive plots: pip install bokeh")    except ImportError as e:    print(f"Note: Visualization libraries not available: {e}")    print("Install required packages: pip install matplotlib seaborn pandas bokeh numpy")except Exception as e:    print(f"Note: Error creating visualizations: {e}")# ============================================================================# STEP 10: Clean up# ============================================================================app.ResetCalculation()print("\n=== Voltage Profile Analysis completed successfully ===")print(f"Results saved in: {script_dir}")